# 검증된 집계 결과 요약

구독형 교육 서비스 전환·리텐션 분석 · 코드잇 중급 1 프로젝트 7팀 · 수행 기간 2026.05–06

이 노트북은 **제공받은 데이터 마트를 재집계한 결과 파일** `results/key_metrics.json`을 읽어 핵심 표를 보여줍니다. 원시 이벤트 분석이나 마트 생성을 다시 수행하지 않으며, 원 프로젝트 노트북의 전체 실행을 검증했다는 의미도 아닙니다. 원시 CSV·계정 연결·네트워크 없이 실행할 수 있습니다.

재집계 로직은 [scripts/reproduce_key_metrics.py](../scripts/reproduce_key_metrics.py), 정의와 제한은 [지표 정의](../docs/methodology.md), 원본 차이는 [검토 기록](../docs/review-notes.md)에 있습니다. 기존 가설과 탐색 과정은 [archive](archive/README.md)에 구분했습니다.

In [1]:
import json
from html import escape
from pathlib import Path

# 저장소 루트 또는 notebooks 폴더에서 실행합니다. 네트워크를 사용하지 않습니다.
root_candidates = [Path.cwd(), Path.cwd().parent]
repo_root = next((p for p in root_candidates if (p / 'results' / 'key_metrics.json').is_file()), None)
if repo_root is None:
    raise FileNotFoundError('저장소 루트 또는 notebooks 폴더에서 실행하고 results/key_metrics.json을 확인하세요.')
metrics = json.loads((repo_root / 'results' / 'key_metrics.json').read_text(encoding='utf-8'))

# IPython이 없는 일반 Python에서도 표준 라이브러리로 표를 출력합니다.
try:
    from IPython.display import HTML, display
except ImportError:
    HTML = display = None

_rendered_tables = []
def show_table(headers, rows):
    plain = '\n'.join([' | '.join(map(str, headers)), ' | '.join(['---'] * len(headers))] + [' | '.join(map(str, row)) for row in rows])
    head = '<thead><tr>' + ''.join('<th>' + escape(str(x)) + '</th>' for x in headers) + '</tr></thead>'
    body = '<tbody>' + ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows) + '</tbody>'
    html = '<table>' + head + body + '</table>'
    _rendered_tables.append({'text/html': html, 'text/plain': plain})
    if display is not None:
        display(HTML(html))
    else:
        print(plain)

print('입력: results/key_metrics.json — 검증된 마트 집계 결과를 읽었습니다.')

입력: results/key_metrics.json — 검증된 마트 집계 결과를 읽었습니다.


## 가입 코호트와 첫 결제

비교 가입 기간은 폐지 전 **2022년 5–11월**, 폐지 후 **2023년 5–11월**입니다. 첫 결제는 가입 후 30일 내 이벤트를 기준으로 합니다. 가입자 수 증가와 전환율 하락을 함께 관찰한 결과이며, 정책 변경의 인과 효과를 분리한 분석은 아닙니다.

In [2]:
cohort_labels = [('pre_2022_05_11', '폐지 전 · 2022.05–11'), ('post_2023_05_11', '폐지 후 · 2023.05–11')]
cohort_rows = []
for key, label in cohort_labels:
    row = metrics['activation_cohorts'][key]
    cohort_rows.append([label, f"{row['users']:,}", f"{row['positive_users']:,}", f"{row['rate_pct']:.2f}%", f"{row['mean_first_paid_amount']:,.2f}원", f"{row['mean_first_discount_amount']:,.2f}원"])
show_table(['가입 코호트', '가입자', '첫 결제자', '첫 결제율', '결제자 평균 첫 결제금액', '결제자 평균 할인금액'], cohort_rows)

가입 코호트,가입자,첫 결제자,첫 결제율,결제자 평균 첫 결제금액,결제자 평균 할인금액
폐지 전 · 2022.05–11,"27,543","2,773",10.07%,"30,347.51원","5,124.92원"
폐지 후 · 2023.05–11,"46,055","2,788",6.05%,"34,957.72원","1,074.41원"


가입자는 27,543명에서 46,055명으로 증가했고 첫 결제율은 10.07%에서 6.05%로 낮아졌습니다. 결제자 평균 첫 결제금액은 증가하고 평균 할인금액은 감소했습니다. 금액과 할인은 결제자 집단의 평균이며 전체 가입자 평균이나 장기 매출이 아닙니다.

## 폐지 후 가입자: 발표용 4개 세그먼트

레슨 완료 수는 **가입 후 7일 미만 범위 중 첫 결제 이전 `complete_lesson` 이벤트 건수**입니다. 고유 레슨 수와 다릅니다. 콘텐츠 경험은 결제 이전의 고유 콘텐츠 또는 가입 후 7일 전체의 조회·시작 여부로 정의합니다. 따라서 조기결제자의 결제 후 조회·시작이 포함될 수 있습니다.

미경험형(기존 표현: 무행동형)은 이 콘텐츠 경험 조건과 레슨 완료가 없는 집단입니다. 후기·질문·결제 페이지 등 모든 활동이 없다는 뜻은 아닙니다.

In [3]:
segment_specs = [
    ('inactive', '미경험형 / 무행동형', '콘텐츠 경험 없음 · 레슨 완료 0회'),
    ('explorer', '탐색형', '콘텐츠 경험 있음 · 레슨 완료 0회'),
    ('weak_learner', '약한 학습형', '레슨 완료 1–6회'),
    ('strong_learner', '강한 학습형', '레슨 완료 7회 이상'),
]
segment_rows = []
for key, label, definition in segment_specs:
    row = metrics['conversion_segments_post'][key]
    segment_rows.append([label, definition, f"{row['users']:,}", f"{row['positive_users']:,}", f"{row['rate_pct']:.2f}%"])
assert sum(x['users'] for x in metrics['conversion_segments_post'].values()) == metrics['activation_cohorts']['post_2023_05_11']['users']
assert sum(x['positive_users'] for x in metrics['conversion_segments_post'].values()) == metrics['activation_cohorts']['post_2023_05_11']['positive_users']
show_table(['세그먼트', '기준', '유저 수', '첫 결제자 수', '30일 내 첫 결제율'], segment_rows)

세그먼트,기준,유저 수,첫 결제자 수,30일 내 첫 결제율
미경험형 / 무행동형,콘텐츠 경험 없음 · 레슨 완료 0회,"21,137",228,1.08%
탐색형,콘텐츠 경험 있음 · 레슨 완료 0회,"12,419","1,363",10.98%
약한 학습형,레슨 완료 1–6회,"6,973",235,3.37%
강한 학습형,레슨 완료 7회 이상,"5,526",962,17.41%


레슨 완료가 없는 두 집단에서도 콘텐츠 경험에 따라 전환율 차이가 관찰됩니다. 레슨 7회 이상은 유저 규모와 결제율을 함께 검토해 선정한 **활성화 후보 기준**입니다. 표는 관찰된 집단의 차이를 보여주며, 특정 사용자를 다음 단계로 이동시키면 같은 폭으로 결제가 증가한다는 뜻은 아닙니다. 온보딩 액션과 기대 매출은 실험 설계·시뮬레이션 단계입니다.

## 갱신: 폐지 후 첫 결제자 중 60일 관찰 가능한 집단

여기서는 `is_post_removed == 1`이고 `can_observe_60d == 1`인 유저만 사용한 집계 결과를 읽습니다. 갱신율의 분자는 `renewed_60d == 1`인 유저 수, 분모는 해당 조건을 만족하는 활동일수 그룹의 유저 수입니다.

활동일수는 제공 마트의 `active_learning_days_postpay_30d` 열을 사용합니다. **최대 32일이 있어 엄밀한 rolling 30일로 확인된 변수는 아닙니다.** 60일 관찰 가능 여부로 모집단을 맞췄다는 사실과 활동일수 산정 경계의 검증은 별개입니다.

In [4]:
retention = metrics['retention_conditions']['post_removal_observable_60d']
retention_rows = []
for key, label in [('0-4', '저활동 · 0–4일'), ('5-13', '중간활동 · 5–13일'), ('14+', '고활동 · 14일 이상 (현재 마트 최대 32일)')]:
    row = retention[key]
    retention_rows.append([label, f"{row['users']:,}", f"{row['positive_users']:,}", f"{row['rate_pct']:.2f}%"])
show_table(['학습 활동일수 그룹', '60일 관찰 가능 유저', '60일 내 갱신 유저', '갱신율'], retention_rows)
checks = metrics['data_checks']
show_table(['확인 항목', '마트 재집계 값'], [
    ['갱신 마트 전체 유저 수', f"{checks['retention_users']:,}"],
    ['위 3개 그룹의 분석 유저 합계', f"{sum(x['users'] for x in retention.values()):,}"],
    ['활동일수 최댓값', f"{checks['max_activity_days']:g}일"],
    ['활동일수 30일 초과 유저 수', f"{checks['activity_days_over_30']:,}"],
])

학습 활동일수 그룹,60일 관찰 가능 유저,60일 내 갱신 유저,갱신율
저활동 · 0–4일,905,440,48.62%
중간활동 · 5–13일,"1,234",737,59.72%
고활동 · 14일 이상 (현재 마트 최대 32일),"1,184",759,64.10%


확인 항목,마트 재집계 값
갱신 마트 전체 유저 수,"5,201"
위 3개 그룹의 분석 유저 합계,"3,323"
활동일수 최댓값,32일
활동일수 30일 초과 유저 수,54


같은 모집단 조건에서 48.62%·59.72%·64.10%가 재집계됩니다. 이는 학습 활동일수와 갱신의 **연관성**입니다. 활동과 갱신의 시간창이 겹칠 수 있어 미래 갱신을 예측한 성능이나 학습 유도의 인과 효과로 읽지 않습니다.

과거 발표의 48.0%·56.9%·63.2% 조합은 이 조건과 같지 않습니다. 전체 기간·관찰 가능 필터를 달리한 결과는 JSON에 별도로 보존했습니다. [검토 기록](../docs/review-notes.md)을 통해 분모와 기준을 함께 확인할 수 있습니다.

## 실행과 해석의 범위

- 이 노트북은 이미 재집계된 JSON을 읽고 표를 표시합니다. 마트에서 직접 다시 계산하려면 별도로 제공 CSV와 `scripts/reproduce_key_metrics.py`를 사용해야 합니다.
- 원시 로그에서 마트를 만드는 전체 과정, 갱신 열의 정확한 시간 경계, 실제 서비스 개선 효과는 이 실행으로 검증하지 않습니다.
- 팀 전체 결과와 개인 기여는 [README](../README.md) 및 [기여 범위](../docs/contribution.md)에 구분합니다.
- 기존 탐색 노트북의 잠정 결론과 시도는 [archive](archive/README.md)에 남기고, 현재 확인된 집계 기준은 이 요약과 README를 우선합니다.